## MININET_TEST

In [2]:
import torch
from torch import nn

In [3]:
x = torch.randn(100, 3)
layer = nn.Linear(3, 5)
print(layer(x).shape)
print(layer.weight)
print(layer.bias)

torch.Size([100, 5])
Parameter containing:
tensor([[ 0.5702, -0.1754, -0.1922],
        [ 0.4425,  0.0187, -0.0541],
        [ 0.0628, -0.4785,  0.1006],
        [-0.1086, -0.1965,  0.2813],
        [-0.5360,  0.0853, -0.2138]], requires_grad=True)
Parameter containing:
tensor([-0.4219, -0.3476,  0.5662,  0.3059, -0.1469], requires_grad=True)


In [ ]:
# relu를 통과하면 음수인 애들이 진짜 사라질까?
x = torch.randn(2, 5)
print(x)
layer = nn.ReLU()
print(layer(x)) # 사라진다!!

tensor([[ 0.3710, -0.8065, -1.1122, -0.1563,  0.5394],
        [ 0.3563, -0.3886, -1.9934,  1.9965, -0.1347]])
tensor([[0.3710, 0.0000, 0.0000, 0.0000, 0.5394],
        [0.3563, 0.0000, 0.0000, 1.9965, 0.0000]])


In [ ]:
x = torch.randn(3, 7)
print(x)
drop = nn.Dropout(p = 0.2) # 논문은 살릴 확률 --> 구현은 죽일 확률로 된다. train과 test시에 다르게 동작한다.
print(drop(x))

tensor([[ 0.2129, -0.1067, -0.6676, -0.5193,  1.7760, -0.8224, -0.9544],
        [ 1.4141, -0.4204,  0.6228, -0.3818,  0.8972, -0.4367, -0.0284],
        [ 0.5522, -0.1712, -0.1737,  0.9328,  1.7288,  0.0262, -0.3631]])
tensor([[ 0.2661, -0.1334, -0.0000, -0.6492,  2.2199, -1.0280, -1.1930],
        [ 0.0000, -0.5255,  0.7785, -0.4773,  0.0000, -0.5458, -0.0355],
        [ 0.6902, -0.0000, -0.2171,  0.0000,  0.0000,  0.0000, -0.4538]])


### Dropout의 두 가지 동작
Dropout은 단순히 값을 0으로 만드는 것만 하지 않아요.
1. 일부 뉴런을 0으로 만든다 (p=0.2 → 20% 확률로 0)
2. 살아남은 값들을 1/(1-p) 배로 스케일 업한다
p=0.2이면 → 1/(1-0.2) = 1/0.8 = 1.25배 곱해줘요.

왜 스케일 업을 할까?
예를 들어 값이 10개 있고 p=0.2라면:

Dropout 적용 시 → 2개가 0이 되고, 8개만 살아남음
근데 그냥 두면 전체 합이 줄어들어버림 (원래 합의 80%밖에 안 됨)
그러면 학습할 때와 추론할 때 값의 스케일이 달라지는 문제 발생

그래서 살아남은 값에 1.25를 곱해서 "기댓값을 원래랑 똑같이" 유지시켜줘요. 

E[dropout(x)] = (1-p) * x * 1/(1-p) = x

기댓값이 x로 보존됨! 

In [6]:
class sample_model(nn.Module):
    def __init__(self):
        super().__init__()
        self.drop_layer = nn.Sequential(nn.Linear(5, 7),
                                        nn.Dropout(p=0.3))
    def forward(self, x):
        x = self.drop_layer(x)
        return x
    
model = sample_model()
model.train() # train mode
x = torch.randn(3, 5)
print(model(x))

model.eval() # test mode
print(model(x))

tensor([[-0.1046,  0.2901,  0.5550,  0.0000, -0.0000, -0.4383,  0.2932],
        [ 0.3994, -0.0058,  0.5518,  1.5063, -1.6802, -0.9664, -0.0259],
        [-0.8771,  0.6148,  0.0000, -0.7782,  0.1385,  0.5573,  1.4613]],
       grad_fn=<MulBackward0>)
tensor([[-0.0732,  0.2031,  0.3885,  0.5633, -0.3563, -0.3068,  0.2052],
        [ 0.2796, -0.0041,  0.3863,  1.0544, -1.1761, -0.6765, -0.0182],
        [-0.6140,  0.4303,  0.4315, -0.5448,  0.0969,  0.3901,  1.0229]],
       grad_fn=<AddmmBackward0>)


In [8]:
layer = nn.Conv2d(in_channels=1, out_channels=2, kernel_size=3, stride=1, padding=1) # stride=1, padding=0 이 디폴트
layer(torch.randn(32, 1, 5, 5)).shape  # 5x5 흑백 사진 32장 -> 개 채 행 열
# nn.Linear(3,5) # 채채 # 얘는 채 또는 개채를 원함, 개 x 3
# nn.Conv2d(3,5) # 채채 # 얘는 채행열 또는 개채행열을 원함, 개 x 3 x 행 x 열

torch.Size([32, 2, 5, 5])

In [10]:
layer = nn.Conv2d(3, 5, 3, stride = 2, padding = 1)
x = torch.randn(32, 3, 5, 5)
print(layer(x).shape)

torch.Size([32, 5, 3, 3])


In [ ]:
conv1 = nn.Conv2d(1,8,6,stride=2)
x=torch.randn(32,1,28,28)
print(conv1(x).shape)

conv2 = nn.Conv2d(8,16,3,padding=1)
print(conv2(conv1(x)).shape)

Maxpool = nn.MaxPool2d(kernel_size=2, stride=(2,2)) # Kernel_size -> pooling할 전체에서의 subset size
print(Maxpool(conv2(conv1(x))).shape)

In [ ]:
maxpool = nn.MaxPool2d(2) # 2로만 줘도 자동 Kernel_size = 2, stride = (2,2), stride --> kernel size랑 동일하게 됨
x = torch.randn(1, 1, 8, 8)
print(x)
print(f"The shape of x {x.shape}")

print(f"Print x after Maxpooling {maxpool(x)}")
print(f"The size of x after maxpooling {maxpool(x).shape}")

tensor([[[[-0.3199, -0.7104,  0.1279,  0.2563, -0.7370, -0.8617, -0.1205,
            1.0294],
          [-1.4976,  0.9563, -0.8779,  0.9538,  0.4869,  0.2580, -0.7144,
            0.7995],
          [ 0.1339, -1.2413, -0.7944,  0.6378,  1.2917,  0.8613, -1.6189,
           -2.2348],
          [ 1.5719,  1.4267, -0.0842, -1.0729,  0.3327, -0.6573,  0.0473,
            0.2653],
          [-0.6084,  0.8086, -0.3914,  0.1670,  0.3385, -0.5156, -0.8732,
           -0.2007],
          [-0.2743,  0.7891,  0.5615,  0.8585,  0.3187, -0.2526, -0.7648,
           -1.1794],
          [-0.4578, -0.4584, -0.5877, -0.8431, -0.0390, -1.2049, -0.5138,
           -0.7831],
          [-0.4716,  0.0722, -0.9267, -0.5166,  0.4752, -0.8452, -0.2886,
            0.9794]]]])
The shape of x torch.Size([1, 1, 8, 8])
Print x after Maxpooling tensor([[[[1.5719, 1.2917],
          [0.8585, 0.9794]]]])
The size of x after maxpooling torch.Size([1, 1, 2, 2])


In [15]:
avgpool = nn.AvgPool2d(2)
x = torch.randn(1, 1, 6, 6)
print(x)
print(avgpool(x))
print(avgpool(torch.randn(32, 3, 6, 6)).shape)


tensor([[[[ 2.0441,  0.7915,  1.0976, -1.1787,  0.3083, -0.3585],
          [ 0.4775, -0.7079, -0.2382, -2.0166, -0.3430,  1.8383],
          [ 0.2555, -0.2983, -0.8738,  0.5000, -0.7790, -0.2456],
          [-0.1666, -0.2336,  1.6232,  1.1316, -1.1740, -2.4889],
          [ 0.3950,  0.7209,  0.0664, -1.7758, -0.8074,  0.5040],
          [ 0.9829, -0.1956,  0.1007,  0.9226, -0.5522, -0.0140]]]])
tensor([[[[ 0.6513, -0.5840,  0.3613],
          [-0.1107,  0.5952, -1.1719],
          [ 0.4758, -0.1715, -0.2174]]]])
torch.Size([32, 3, 3, 3])


In [26]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv1 = nn.Conv2d(1, 8, 6, stride = 2)
        self.conv2 = nn.Conv2d(8, 16, 3, padding =1)
        self.Maxpool2 = nn.MaxPool2d(2)
        self.fc = nn.Linear(16 * 6 * 6,10)
        
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.Maxpool2(x)
        x = torch.flatten(x, start_dim=1)
        x = self.fc(x)
        return x

x = torch.randn(32, 1, 28, 28)
model = CNN()
print(model(x).shape)
        
        

torch.Size([32, 10])


In [25]:
x = torch.randn(32, 2, 8, 8)
print(torch.flatten(x, start_dim=0).shape) # 0번째 차원부터 끝까지 전부 하나로 펼친다..
print(torch.flatten(x, start_dim=1).shape) # 1번째 차원부터 끝까지 전부 하나로 펼친다..


torch.Size([4096])
torch.Size([32, 128])
